In [0]:
%run ./config

In [0]:
agg_df=spark.read.format('delta').load(gold+'/fact_table')
repayment_df=spark.read.format('delta').load(gold+'/repayment')

In [0]:
agg_df.createOrReplaceTempView('agg_df')
repayment_df.createOrReplaceTempView('repayments_df')

In [0]:
early_warning_indicators= spark.sql(
    """with repayments_with_branch as (
    select 
        r.due_date,
        f.branch_id,
        r.amount_due,
        r.amount_paid,
        r.days_past_due
    from repayments_df r
    join agg_df f on r.loan_id = f.loan_id
),

daily_branch_aggregates as (
    select 
        due_date,
        branch_id,
        sum(amount_due) as daily_due,
        sum(amount_paid) as daily_paid,
        avg(days_past_due) as avg_dpd,
        case 
            when sum(amount_due) > 0 then (sum(amount_due) - sum(amount_paid)) / sum(amount_due) 
            else 0.0 
        end as repayment_slippage_rate
    from repayments_with_branch
    group by due_date, branch_id
),

rolling_metrics as (
    select
        due_date,
        branch_id,
        repayment_slippage_rate,
        avg_dpd,
        avg(avg_dpd) over (
            partition by branch_id 
            order by due_date 
            range between interval 7 days preceding and interval 1 day preceding
        ) as trailing_7d_avg_dpd
    from daily_branch_aggregates
)

select
    due_date as date,
    branch_id,
    round(repayment_slippage_rate, 4) as repayment_slippage_rate,
    round(coalesce(avg_dpd - trailing_7d_avg_dpd, 0.0), 2) as avg_dpd_change_vs_7_day_avg,
    case 
        when repayment_slippage_rate > 0.15 or (avg_dpd - trailing_7d_avg_dpd) > 3.0 then 'high'
        when repayment_slippage_rate > 0.05 or (avg_dpd - trailing_7d_avg_dpd) > 1.0 then 'medium'
        else 'low'
    end as risk_flag
from rolling_metrics
where due_date between '2024-04-01' and '2024-04-30'
order by date desc, branch_id 
                                    """).display()

In [0]:
early_warning_indicators= spark.sql("""
                                    with cte1 as (
                                        select  branch_id, 
                                                round(sum(outstanding_amount),2) outstanding_amount,
                                                round(sum(principal),2) disburded_total , 
                                                round(avg(urs_score),2) avg_urs_score
                                        from agg_df
                                        group by branch_id
                                    ),
                                    cte2 as (
                                        select  branch_id, 
                                                r.loan_id,
                                                days_past_due,
                                                amount_due,due_date,
                                                amount_paid, paid_date
                                        from repayments_df r join agg_df a on r.loan_id = a.loan_id
                                    )
                                    select due_date, branch_id,
                                           round(sum(amount_due),2) total_amount_due, 
                                           round(sum(amount_paid),2) total_amount_paid,
                                           round(round(sum(amount_paid),2)*100/
                                           round(sum(amount_due),2),2) repayment_rate,
                                           avg(days_past_due) over(partition by branch_id order by due_date range between 7 preceding and current row)
                                    from cte2 
                                    where due_date between '2024-04-01' and '2024-04-30'
                                    group by branch_id, due_date
                                    
                                    """).display()